### 01 - Instalação (Bibliotecas)

In [ ]:
%pip install pandas

%pip install numpy

%pip install matplotlib

%pip install seaborn

!pip install torch torchvision

### 02 - Recursos (Importações)

In [ ]:
from collections import Counter

import math

import pandas as pd

import numpy as np

import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import DataLoader, TensorDataset

import seaborn as sbn

### 03 - Classes

In [ ]:
class PedagocicalSelfAttention(nn.Module):
  def __init__(self, embedding_size):
    super(PedagocicalSelfAttention, self).__init__()

    self.embedding_size = embedding_size

    self.values = nn.Linear(embedding_size, embedding_size, bias=False)
    self.keys = nn.Linear(embedding_size, embedding_size, bias=False)
    self.queries = nn.Linear(embedding_size, embedding_size, bias=False)

  def forward(self, x):
    V = self.values(x)
    K = self.keys(x)
    Q = self.queries(x)

    score = torch.bmm(Q, K.transpose(1,2))
    score = score / math.sqrt(self.embedding_size)

    attention_weights = F.softmax(score, dim=-1)

    output = torch.bmm(attention_weights, V)

    return output, attention_weights

class TransformerSpamClassifier(nn.Module):
  def __init__(self, vocabulary_size, embedding_size, max_length):
    super(TransformerSpamClassifier, self).__init__()

    self.token_embedding = nn.Embedding(vocabulary_size, embedding_size, padding_idx=0)

    self.position_embedding = nn.Embedding(max_length, embedding_size)

    self.attention = PedagocicalSelfAttention(embedding_size)

    self.fc = nn.Linear(embedding_size, 2)

  def forward(self, x):
    N, sequence_length = x.shape

    positions = torch.arange(0, sequence_length).expand(N, sequence_length).to(x.device)

    output = self.token_embedding(x) + self.position_embedding(positions)

    output, attention_weights = self.attention(output)

    pooled = output.mean(dim=1)

    logits = self.fc(pooled)

    return logits, attention_weights

### 04 - Funções

In [ ]:
def encode_tokens(tokens, word_to_index, max_sequence_length):
  encoded_tokens = [word_to_index.get(token, word_to_index["<UNK>"]) for token in tokens]

  if (len(encoded_tokens) < max_sequence_length):
    encoded_tokens += [word_to_index["<PAD>"]] * (max_sequence_length - len(encoded_tokens))

  return encoded_tokens

def explain_predicition(text: str, model: TransformerSpamClassifier, index_to_word, word_to_index, max_sequence_length):
  model.eval()

  tokens = text.lower().split()[:MAX_SEQUENCE_LENGTH]

  encoded = encode_tokens(tokens, word_to_index, max_sequence_length)

  input_tensor = torch.tensor([encoded], dtype=torch.long).to(device)

  real_tokens = [index_to_word[index] for index in encoded if index != 0]

  sequence_length = len(real_tokens)

  with torch.no_grad():
    logits, attention = model(input_tensor)

    predicition_class = torch.argmax(logits, dim=1).item()

    confidence = F.softmax(logits, dim=1)[0][predicition_class].item()

  attention_matrix = attention[0, :sequence_length, :sequence_length].cpu().numpy()

  class_name = "SPAM" if predicition_class == 1 else "HAM (Comum)"

  print(f"\nMensagem: {text}")

  print(f"\nPrevisão do Modelo: {class_name} ({(confidence * 100):.2f}% de Confiança)\n")

  plt.figure(figsize=(8,6))

  sbn.heatmap(attention_matrix, annot=True, cmap="Reds", fmt=".2f", xticklabels=real_tokens, yticklabels=real_tokens)

  plt.title("Mapa de Auto-Atenção (Self-Attention)")

  plt.xlabel("Token de Destino")
  plt.ylabel("Token de Origem")

  plt.xticks(rotation=45)
  plt.yticks(rotation=0)

  plt.show()

### 05 - Implementação

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

try:
  dataframe = pd.read_csv("./Data/spam_utf8.csv", encoding="utf8")

  # Extraindo colunas específicas.

  dataframe = dataframe[["v1", "v2"]] # Label e Text.

  dataframe.columns = ["label", "text"]

  # Conversão numérica.

  dataframe["label"] = dataframe["label"].map({ "ham": 0, "spam": 1 })

  print(f"Exemplo de Saída ({len(dataframe)} Registros):\n")

  display(dataframe.head())

  MAX_SEQUENCE_LENGTH = 20

  VOCABULARY_SIZE = 3000

  EMBEDDING_SIZE = 32

except:
  print("Erro ao carregar o CSV!")

  exit()

### 06 - Tokenização

In [ ]:
dataframe["tokens"] = dataframe["text"].apply(lambda token: str(token).lower().split()[:MAX_SEQUENCE_LENGTH])

all_words = [token for tokens in dataframe["tokens"] for token in tokens]

vocabulary = Counter(all_words)

common_tokens = [token for (token, _) in vocabulary.most_common(VOCABULARY_SIZE - 2)]

token_to_index = { token: index + 2 for (index, token) in enumerate(common_tokens) }

token_to_index["<PAD>"] = 0
token_to_index["<UNK>"] = 1

index_to_word = { index: token for (token, index) in token_to_index.items() }

X = torch.tensor([encode_tokens(tokens, token_to_index, MAX_SEQUENCE_LENGTH) for tokens in dataframe["tokens"]], dtype=torch.long)
y = torch.tensor(dataframe["label"].values, dtype=torch.long)

dataset = TensorDataset(X, y)

dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

### 07 - Treinamento e Teste

In [ ]:
model = TransformerSpamClassifier(VOCABULARY_SIZE, EMBEDDING_SIZE, MAX_SEQUENCE_LENGTH).to(device)

print(f"Modelo: {model}")

print("\nTreinamento:")

optimizer = optim.Adam(model.parameters(), lr=0.001)

criterion = nn.CrossEntropyLoss()

EPOCHS = 100

for epoch in range(0, EPOCHS):
  model.train()

  total_loss = 0

  correct = 0

  total = 0

  for batch_X, batch_y in dataloader:
    batch_X, batch_y = batch_X.to(device), batch_y.to(device)

    optimizer.zero_grad()

    logits, _ = model(batch_X)

    loss = criterion(logits, batch_y)

    loss.backward()

    optimizer.step()

    total_loss += loss.item()

    predictions = torch.argmax(logits, dim=1)

    correct += (predictions == batch_y).sum().item()

    total += batch_y.size(0)

  accuracy = (100 * correct) / total

  print(f"\nÉpoca {epoch + 1} de {EPOCHS} | Perda: {(total_loss / len(dataloader)):.4f} | Acurácia: {accuracy:.2f}%")

print("\nTreinamento Concluído.")

### 08 - Aplicação Prática

In [ ]:
span_test = "URGENT YOU WON A FREE PRIZE RING TO THIS NUMBER TO CLAIM MONEY"

explain_predicition(span_test, model, index_to_word, token_to_index, MAX_SEQUENCE_LENGTH)